<a href="https://colab.research.google.com/github/emmaenglishwilkins/IAC-Implementation/blob/main/reconCompare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Comparing PROD to UAT
where uat is the most updated version

In [ ]:
from google.colab import files
import pandas as pd

In [ ]:
uploaded = files.upload()
report = next(iter(uploaded))
# report = 'LATESTcomvest_report.xls'
xls = pd.ExcelFile(report)
print(xls.sheet_names)

# prod = pd.read_excel(report, "Custodian Account_PRD")
# uat  = pd.read_excel(report, "Custodian Account_UAT")
# prod      = pd.read_excel(report, "Payee_PRD")
# uat       = pd.read_excel(report, "Payee_UAT")

# prod = pd.read_excel(report, "prod")
# uat  = pd.read_excel(report, "uat")

# prod = pd.read_excel(report, "PROD")
# uat  = pd.read_excel(report, "UAT")

prod = pd.read_excel(report, "HF")
uat  = pd.read_excel(report, "PE")

Saving NUB_Custodians.xlsx to NUB_Custodians.xlsx
['HF', 'PE']


In [ ]:
prod.columns

Index(['Custodian Code', 'Custodian Name', 'Legal Entity Code',
       'Legal Entity Name', 'Legal Entity Type', 'Region', 'Industry Sector',
       'Exposure Ticker', 'Asset', 'Classification', 'Location', 'Owner',
       'Is Active', 'Has SSIs', 'Has STIs', 'Has Mappings', 'Has Pending SSI',
       'Has Pending STI', 'Has Pending Mapping', 'Created By', 'Created On',
       'Last Modified By', 'Last Modified On'],
      dtype='object')

In [ ]:
# unique identifer - Account
# columns to ignore - 'Account Info Last Modified By', 'Account Info Last Modified On' 'SSILast Modified On', 'SSILast Modified By'
# key_col = 'Account'  # whatever column in both identifies the row
# ignore_cols = ['Account Info Last Modified By', 'Account Info Last Modified On', 'SSILast Modified On', 'SSILast Modified By', key_col]

# key_col = 'LegalEntityName'  # whatever column in both identifies the row
# ignore_cols = ['RootLegalEntityId', 'ParentLegalEntityId', key_col]

key_col = 'Custodian Code'
ignore_cols = ['Created By', 'Created On','Last Modified By', 'Last Modified On',key_col] # had to add key col to ignore too

# key_col = 'Account Number'
# ignore_cols = [key_col]
common_cols = [c for c in uat.columns if c in prod.columns and c not in ignore_cols]

In [ ]:
# merged = uat.merge(prod, how="left", on=key_col, suffixes=("_UAT", "_PROD"))
merged = uat.merge(prod, how="left", on=key_col, suffixes=("_PE", "_HF"))

In [ ]:
# export merge file - this was for debugging but the merge looks good here
# merged.to_excel("merged.xlsx", index=False)

In [ ]:
def compare_row(row):
    mismatch_cols = []
    for col in common_cols:
        uat_val = row[f"{col}_PE"]
        prod_val = row[f"{col}_HF"]
        if pd.isna(uat_val) and pd.isna(prod_val):
            continue
        if uat_val != prod_val:
            mismatch_cols.append(col)
    row["MatchStatus"] = "mismatch" if mismatch_cols else "match"
    row["MismatchCols"] = mismatch_cols
    return row


In [ ]:
merged = merged.apply(compare_row, axis=1)

In [ ]:
# Highlight PROD cells based on mismatch columns - uat is truth
# def highlight_row(row):
#   styles = [''] * len(row)
#   if row["MatchStatus"] == "mismatch":
#     for col in row["MismatchCols"]:
#       print(f'getting column for {col}')
#       prod_idx = merged.columns.get_loc(f"{col}_PROD")
#       styles[prod_idx] = "background-color: yellow"
#   return styles


In [ ]:
# highlight UAT cells based on mismatch columns
def highlight_row(row):
  styles = [''] * len(row)
  if row["MatchStatus"] == "mismatch":
    for col in row["MismatchCols"]:
      print(f'getting column for {col}')
      prod_idx = merged.columns.get_loc(f"{col}_PE")
      styles[prod_idx] = "background-color: yellow"
  return styles

In [ ]:
# find missing accounts in PROD
missing_in_prod = uat.loc[~uat[key_col].isin(prod[key_col]), key_col]

# missing in UAT (not always needed but whatever)
missing_in_uat = prod.loc[~prod[key_col].isin(uat[key_col]), key_col]

In [ ]:
merged.head(10)

,Account Code,Portfolio Code_PE,Custodian Code_PE,Account Type Code_PE,Margin?_PE,OTC?_PE,IsABF?_PE,IsActive?_PE,IsApproved_PE,Has Pending SSI_PE,...,IsActive?_HF,IsApproved_HF,Has Pending SSI_HF,Has Mappings_HF,Is Collateral,Is Longbox,Is Third Party,Is Triparty,MatchStatus,MismatchCols
0,#7ENKGEJ,NBPIHUL_0003,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
1,#7ENKH6W,NBPDFIL_0001,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
2,#7ENKHYQ,NBPDFILS,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
3,#SEA4N78,ZZZ,WF,Default,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
4,#SEA4O2E,ZZZ,WF,Default,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
5,(EEAJ7CQ,NBPIHUL,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
6,(EEAJWYB,NBAESNPECO,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
7,(EEAJXQ3,NBPECOFL,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
8,(EEAK0V3,NBPECOFTL,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."
9,(EEAK1LS,NBPECOFTL,WF,CASH,False,1.0,False,True,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mismatch,"[Portfolio Code, Custodian Code, Account Type ..."


In [ ]:
ordered_cols = [key_col]
for col in common_cols:
    ordered_cols.append(f"{col}_PE")
    ordered_cols.append(f"{col}_HF")

# Add the 'MatchStatus' and 'MismatchCols' to the end
ordered_cols.append("MatchStatus")
ordered_cols.append("MismatchCols")

# Reindex the DataFrame with the new column order
merged = merged[ordered_cols]

display(merged.head())

,Custodian Code,Custodian Name_PE,Custodian Name_HF,Legal Entity Code_PE,Legal Entity Code_HF,Legal Entity Name_PE,Legal Entity Name_HF,Legal Entity Type_PE,Legal Entity Type_HF,Region_PE,...,Has Mappings_PE,Has Mappings_HF,Has Pending SSI_PE,Has Pending SSI_HF,Has Pending STI_PE,Has Pending STI_HF,Has Pending Mapping_PE,Has Pending Mapping_HF,MatchStatus,MismatchCols
0,BAML,BAML,"BANK OF AMERICA, N.A., CHARLOTTE,NC",NeubergerWireFundAdmin,NaN,NeubergerWireFundAdmin,NaN,Prime Broker,NaN,NaN,...,False,False,False,False,False,False,False,False,mismatch,"[Custodian Name, Legal Entity Code, Legal Enti..."
1,BankOzk,BankOzk,NaN,NeubergerWireFundAdmin,NaN,NeubergerWireFundAdmin,NaN,Prime Broker,NaN,America,...,False,NaN,False,NaN,False,NaN,False,NaN,mismatch,"[Custodian Name, Legal Entity Code, Legal Enti..."
2,BARC,BARC,Barclays Bank PLC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,mismatch,[Custodian Name]
3,BBH,BBH,Brown Brother Harriman,NeubergerWireFundAdmin,NaN,NeubergerWireFundAdmin,NaN,Prime Broker,NaN,Europe,...,False,False,False,False,False,False,False,False,mismatch,"[Custodian Name, Legal Entity Code, Legal Enti..."
4,BFF,BFF Bank S.p.A.,BFF Bank S.p.A.,NaN,NaN,NaN,NaN,NaN,NaN,Europe,...,False,False,False,False,False,False,False,False,mismatch,[Region]


In [ ]:
with pd.ExcelWriter("comparison_output.xlsx", engine="openpyxl") as writer:
    merged.style.apply(highlight_row, axis=1).to_excel(writer, sheet_name="Comparison", index=False)
    missing_in_uat.to_frame(name="MissingInPE").to_excel(writer, sheet_name="MissingInPE", index=False)
    missing_in_prod.to_frame(name="MissingInHF").to_excel(writer, sheet_name="MissingInHF", index=False)

getting column for Custodian Name
getting column for Legal Entity Code
getting column for Legal Entity Name
getting column for Legal Entity Type
getting column for Custodian Name
getting column for Legal Entity Code
getting column for Legal Entity Name
getting column for Legal Entity Type
getting column for Region
getting column for Is Active
getting column for Has SSIs
getting column for Has STIs
getting column for Has Mappings
getting column for Has Pending SSI
getting column for Has Pending STI
getting column for Has Pending Mapping
getting column for Custodian Name
getting column for Custodian Name
getting column for Legal Entity Code
getting column for Legal Entity Name
getting column for Legal Entity Type
getting column for Region
getting column for Has STIs
getting column for Region
getting column for Custodian Name
getting column for Legal Entity Code
getting column for Legal Entity Name
getting column for Legal Entity Type
getting column for Is Active
getting column for Has SS

In [ ]:
# print unique values in MissmatchCols
merged["MismatchCols"].drop_duplicates()


,MismatchCols
0,[]
33,"[Company Short Name, Payee Name, Address1, Cit..."
38,[Portfolio]
43,"[Company Short Name, Payee Name, Address1, Cit..."
44,"[Address1, City, State, Zip]"
63,[SWIFTField72]
100,"[Address1, City, State, Zip, Portfolio]"
106,"[Portfolio, SWIFTField72]"
287,"[Company Short Name, Payee Name, Address1, Add..."
1221,"[Company Short Name, Payee Name, Address1, Add..."


In [ ]:
# count missmatch cols
merged["MismatchCols"].value_counts()

,count
MismatchCols,
[],418
"[Company Short Name, Portfolio, Custodian, Bank Name, Bank BIC, Account Name, Account #, Is Active, Account Status, Currency, Institution Bank Name, Institution BIC, Institution ABA, SSIStatus]",15
"[PBAccount, PBAccount Number]",8
"[Company Short Name, Portfolio, Custodian, Bank Name, Bank BIC, Account Name, Account #, Is Active, Account Status, Currency, Institution Bank Name, Institution ABA, Institution IBANOr Bank Acct No, SWIFTField72, SSIStatus]",2
"[Company Short Name, Portfolio, Custodian, Bank Name, Account Name, Account #, Is Active, Account Status, Currency, Correspondent Bank Name, Institution ABA, Institution IBANOr Bank Acct No, SSIStatus]",1
"[Company Short Name, Portfolio, Custodian, Bank Name, Account Name, Account #, Is Active, Account Status, Currency, Institution Bank Name, Institution ABA, SSIStatus]",1


In [ ]:
merged["MatchStatus"].value_counts()

,count
MatchStatus,
match,418
mismatch,27


In [ ]:
df = pd.read_excel("comparison_output.xlsx", sheet_name="Comparison")

In [ ]:
# Create a list of columns in the desired order (key_col, then UAT/PROD pairs)
ordered_cols = [key_col]
for col in common_cols:
    ordered_cols.append(f"{col}_UAT")
    ordered_cols.append(f"{col}_PROD")

# Add the 'MatchStatus' and 'MismatchCols' to the end
ordered_cols.append("MatchStatus")
ordered_cols.append("MismatchCols")

# Reindex the DataFrame with the new column order
df = df[ordered_cols]

display(df.head())

,Payee Code,Company Short Name_UAT,Company Short Name_PROD,Payee Name_UAT,Payee Name_PROD,Address1_UAT,Address1_PROD,Unnamed: 4_UAT,Unnamed: 4_PROD,Address2_UAT,...,Institution Sort Code_UAT,Institution Sort Code_PROD,Institution IBANOr Bank Acct No_UAT,Institution IBANOr Bank Acct No_PROD,SWIFTField72_UAT,SWIFTField72_PROD,SSIStatus_UAT,SSIStatus_PROD,MatchStatus,MismatchCols
0,I00002_CCP II LP,Comvest Partners,Comvest Partners,2005 Scarpa Family Trust,2005 Scarpa Family Trust,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2005 Scarpa Family Trust 6045-9621,2005 Scarpa Family Trust 6045-9621,Approved,Approved,match,[]
1,I00002_CIP IV LP,Comvest Partners,Comvest Partners,2005 Scarpa Family Trust,2005 Scarpa Family Trust,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2005 Scarpa Family Trust 6045-9621,2005 Scarpa Family Trust 6045-9621,Approved,Approved,match,[]
2,I00016_CIP IV LP,Comvest Partners,Comvest Partners,Athene Annuity and Life Company,Athene Annuity and Life Company,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,214450,214450,Approved,Approved,match,[]
3,I00017_CCP II Int Cay,Comvest Partners,Comvest Partners,Azeez Foundation,Azeez Foundation,NaN,2187 Marseilles Drive,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,FFC: Azeez Foundation 7275-1083,FFC: Azeez Foundation 72751083,Approved,Approved,mismatch,"['Address1', 'City', 'State', 'Zip', 'SWIFTFie..."
4,I00018_CIP IV-A LP,Comvest Partners,Comvest Partners,B&S Opportunity 2007 (Delaware) LP,B&S Opportunity 2007 (Delaware) LP,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Approved,Approved,match,[]


In [ ]:
with pd.ExcelWriter("comp_final.xlsx", engine="openpyxl") as writer:
    df.style.apply(highlight_row, axis=1).to_excel(writer, sheet_name="Comparison", index=False)

IndexError: At least one sheet must be visible

FULL PY FILE: quick run, no debugg or visability
pro: faster
con: less readable

In [ ]:
# from google.colab import files
# import pandas as pd

uploaded = files.upload()
report = next(iter(uploaded))

prod = pd.read_excel(report, "prod")
uat = pd.read_excel(report, "uat")

key_col = 'Account Number'
ignore_cols = [key_col]
common_cols = [c for c in uat.columns if c in prod.columns and c not in ignore_cols]

merged = uat.merge(prod, how="left", on=key_col, suffixes=("_UAT", "_PROD"))

# Compare and mark mismatches and update PROD cells with "UAT | PROD"
def compare_and_update(row, sep=" | "):
    mismatch_cols = []
    for col in common_cols:
        uat_val = row[f"{col}_UAT"]
        prod_val = row[f"{col}_PROD"]

        if pd.isna(uat_val) and pd.isna(prod_val):
            continue
        if uat_val != prod_val:
            mismatch_cols.append(col)
            row[f"{col}_UAT"] = f"{prod_val}{sep}{uat_val}"
    row["MatchStatus"] = "mismatch" if mismatch_cols else "match"
    row["MismatchCols"] = mismatch_cols
    return row

merged = merged.apply(compare_and_update, axis=1)

# Highlight PROD cells based on mismatch columns
def highlight_row(row):
    styles = [''] * len(row)
    if row["MatchStatus"] == "mismatch":
        for col in row["MismatchCols"]:
            uat_idx = merged.columns.get_loc(f"{col}_UAT")
            styles[uat_idx] = "background-color: yellow"
    return styles

# Find missing accounts in PROD
missing_accounts = prd.loc[~prd[key_col].isin(uat[key_col]), key_col]


# Save to Excel
with pd.ExcelWriter("comparison_output.xlsx", engine="openpyxl") as writer:
  # only write _UAT columns
    uat_cols = [col for col in merged.columns if col.endswith("_UAT")] + [key_col] + ["MatchStatus", "MissmatchCols"]
    merged.style.apply(highlight_row, axis=1).to_excel(writer, sheet_name="Comparison", index=False)
    missing_accounts.to_frame(name="MissingInUAT").to_excel(writer, sheet_name="MissingInUAT", index=False)


In [ ]:
'''
Comparing missing information in UAT where PROD is the most updated - ensuring clean testing environment
'''

# import pandas as pd
# from google.colab import files

# uploaded = files.upload()
report = next(iter(uploaded))

prod = pd.read_excel(report, "prod")
uat = pd.read_excel(report, "uat")

key_col = 'Account Code'
ignore_cols = [key_col]
common_cols = [c for c in uat.columns if c in prod.columns and c not in ignore_cols]

# optional deduplication of UAT on key
uat = uat.drop_duplicates(subset=[key_col])

merged = uat.merge(prod, how="left", on=key_col, suffixes=("_UAT", "_PROD"))

def compare_and_update(row):
    mismatch_cols = []
    missing_cols = []
    for col in common_cols:
        uat_val = row[f"{col}_UAT"]
        prod_val = row[f"{col}_PROD"]

        # if both empty, skip
        if pd.isna(uat_val) and pd.isna(prod_val):
            continue

        # case 1: missing in UAT but exists in PROD
        if pd.isna(uat_val) and not pd.isna(prod_val):
            missing_cols.append(col)
        # case 2: mismatch
        elif uat_val != prod_val:
            mismatch_cols.append(col)

    if missing_cols:
        row["MatchStatus"] = "missing_in_UAT"
    elif mismatch_cols:
        row["MatchStatus"] = "mismatch"
    else:
        row["MatchStatus"] = "match"

    row["MismatchCols"] = ",".join(mismatch_cols + missing_cols)
    return row

merged = merged.apply(compare_and_update, axis=1)

def highlight_row(row):
    styles = [''] * len(merged.columns)
    if row["MatchStatus"] in ("mismatch", "missing_in_UAT"):
        for col in row["MismatchCols"].split(","):
            if col:
                uat_idx = merged.columns.get_loc(f"{col}_UAT")
                styles[uat_idx] = "background-color: yellow"
    return styles

missing_in_uat = prod.loc[~prod[key_col].isin(uat[key_col]), key_col]
missing_in_prod = uat.loc[~uat[key_col].isin(prod[key_col]), key_col]

with pd.ExcelWriter("comparison_output.xlsx", engine="openpyxl") as writer:
    merged.style.apply(highlight_row, axis=1).to_excel(writer, sheet_name="Comparison", index=False)
    missing_in_uat.to_frame(name="MissingInUAT").to_excel(writer, sheet_name="MissingInUAT", index=False)
    missing_in_prod.to_frame(name="MissingInPROD").to_excel(writer, sheet_name="MissingInPROD", index=False)



In [ ]:
# find unique missmatch columns

from google.colab import files
import pandas as pd

uploaded = files.upload()
report = next(iter(uploaded))

df = pd.read_excel(report, "comp")

Saving comparison_output (12).xlsx to comparison_output (12).xlsx


In [ ]:
df.columns

Index(['Company Short Name_UAT', 'Payee Code', 'Payee Name_UAT',
       'Address1_UAT', 'Unnamed: 4_UAT', 'Address2_UAT', 'City_UAT',
       'State_UAT', 'Zip_UAT', 'Country_UAT', 'Portfolio_UAT',
       'Account Name_UAT', 'Account #_UAT', 'Default GLAccount Code_UAT',
       'Contact Phone_UAT', 'Contact Email_UAT', 'Contact Name_UAT',
       'Bank Name_UAT', 'Is Active_UAT', 'Payee Status_UAT',
       'Payee Last Modified By_UAT', 'Payee Last Modified On_UAT',
       'TMType Name_UAT', 'Currency_UAT', 'Correspondent Bank Name_UAT',
       'Correspondent Bank Bic_UAT', 'Correspondent Bank Sort Code_UAT',
       'Correspondent Bank ABA_UAT', 'Correspondent Bank Iban Or Acct_UAT',
       'Intermediary Bank Name_UAT', 'Intermediary BIC_UAT',
       'Intermediary ABA_UAT', 'Intermediary Sort Code_UAT',
       'Intermediary Bank IBANor Acct_UAT', 'Institution Bank Name_UAT',
       'Institution BIC_UAT', 'Institution ABA_UAT',
       'Institution Sort Code_UAT', 'Institution IBANOr Bank A

In [ ]:
# unique values in 'MismatchCols'
unique_values = df['MismatchCols'].str.split(',', expand=True).stack().unique()
print(unique_values)

['[]' "['Account #']" "['Is Active']" "['Portfolio']" "['SWIFTField72']"
 "['Zip']" "['Portfolio'" " 'Account #']" "['Company Short Name'"
 " 'Payee Name'" " 'Country'" " 'Portfolio'" " 'Account Name'"
 " 'Account #'" " 'Bank Name'" " 'Is Active'" " 'Payee Status'"
 " 'Currency'" " 'Intermediary Bank Name'" " 'Intermediary BIC'"
 " 'Institution Bank Name'" " 'Institution BIC'" " 'SSIStatus']"
 "['State'" " 'Zip'" " 'Portfolio']" "['Country'" " 'Institution ABA'"
 " 'SWIFTField72'" " 'Address1'" " 'Intermediary ABA'"
 " 'Institution Sort Code'" " 'Institution IBANOr Bank Acct No'"
 "['Address1'" " 'City'" " 'State'"]


In [ ]:
# output df to only have unique columns _UAT and _PRD and key_col = "payee code"


In [ ]:
# Extract unique mismatch columns from the 'MismatchCols' column, excluding empty strings and brackets
mismatch_cols = df['MismatchCols'].str.findall(r"'([^']+)'").explode().dropna().unique().tolist()

# Create a list of columns to keep: the key column and the UAT/PROD pairs for mismatch columns
columns_to_keep = ['Payee Code']
for col in mismatch_cols:
    columns_to_keep.append(f"{col}_UAT")
    columns_to_keep.append(f"{col}_PROD")

# Create a new DataFrame with only the selected columns
mismatch_df = df[df['MatchStatus'] == 'mismatch'][columns_to_keep].copy()

display(mismatch_df.head())

,Payee Code,Account #_UAT,Account #_PROD,Is Active_UAT,Is Active_PROD,Portfolio_UAT,Portfolio_PROD,SWIFTField72_UAT,SWIFTField72_PROD,Zip_UAT,...,Address1_UAT,Address1_PROD,Intermediary ABA_UAT,Intermediary ABA_PROD,Institution Sort Code_UAT,Institution Sort Code_PROD,Institution IBANOr Bank Acct No_UAT,Institution IBANOr Bank Acct No_PROD,City_UAT,City_PROD
29,I00039_CIP IV LP,004776836656,4776836656,Active,Active,CIP IV LP,CIP IV LP,NaN,NaN,10038,...,222 Broadway,222 Broadway,NaN,NaN,NaN,NaN,NaN,NaN,New York,New York
70,I00082_CCP II Assoc GP Comm,003435410660,3435410660,Active,Active,CCP II Assoc GP Comm,CCP II Assoc GP Comm,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,I00089_CIP IV GP Carry,1110017146967,1110017146967,Active,Active,CIP IV GP Carry,CIP IV GP Carry,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117,I00121_CIP IV Co-Invest,0005241395601,5241395601,Active,Active,CIP IV Co-Invest,CIP IV Co-Invest,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,I00143_CCP IV GP Carry,000010647304,10647304,Active,Active,CCP IV GP Carry,CCP IV GP Carry,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# save missmatch df to csv
mismatch_df.to_csv("mismatch_df.csv", index=False)

In [ ]:
print(mismatch_cols)

# should do the output to only have these files in the future

['Account #', 'Is Active', 'Portfolio', 'SWIFTField72', 'Zip', 'Company Short Name', 'Payee Name', 'Country', 'Account Name', 'Bank Name', 'Payee Status', 'Currency', 'Intermediary Bank Name', 'Intermediary BIC', 'Institution Bank Name', 'Institution BIC', 'SSIStatus', 'State', 'Institution ABA', 'Address1', 'Intermediary ABA', 'Institution Sort Code', 'Institution IBANOr Bank Acct No', 'City']
